Riservatezza $\neq$ Integrità:
- La **riservatezza** (confidentiality) impedisce che un estraneo legga il contenuto di un messaggio
- L'**integrità** permette al destinatario di accorgersi se qualcuno ha modificato il messaggio o ha costruito un messaggio falso a nome del mittente.

Abbiamo visto che il one-time pad garantisce una riservatezza fortissima, ma un attaccante può cambiare bit del ciphertext e provocare cambiamenti prevedibili del plaintext. In generale quindi **la cifratura non implica l'integrità**.

### MAC Message Authentication Code
Un **Message Authentication Code (MAC)** usa una chiave segreta $K$ condivisa tra mittente e destinatario. Il mittente prende il messaggio $M$ e calcola un breve valore, detto **tag**, per poi inviare entrambi:
$$T = \text{MAC}_K(M) \quad \text{invia } (M, T)$$
Il destinatario, che conosce $K$, ricalcola il tag sul messaggio ricevuto e lo confronta con quello ricevuto:
$$Verify_K(M, T) = \begin{cases} \text{accept} & \text{se } \text{MAC}_K(M) = T \\ \text{reject} & \text{altrimenti} \end{cases}$$
In generale un MAC serve anche quando si vuole lasciare il messaggio in chiaro (l'importante è che $K$ sia segreto), ciò che vuole garantire è infatt l'integrità.

Per far sì che un MAC sia sicuro, si vuole che:
- la funzione che calcola il tag sia **crittograficamente sicura** (ossia intuitivamente che non sia possibile calcolare il tag senza conoscere la chiave etc...). Tra queste funzioni figurano le funzioni hash crittografiche
- deve usare una **chiave segreta**. Un normale checksum senza chiave non basta

Tra gli attacchi che blocca un MAC sicuro:
- **Modifica di un messaggio intercettato (man in the middle)**: ipotizziamo un attaccante intercetti $(M, T)$ e sostituisca $M$ con $M'$. Se inoltra il vecchio tag $T$, il destinatario calcola $\text{MAC}_K(M')$, che non corrisponderà a $T$. Per far accettare il messaggio modificato dovrebbe produrre un tag valido per $M'$ ma senza conoscere $K$, e ciò è computazionalmente difficile.
- **Creazione di un messaggio falso (message spoofing)**: l'attaccante potrebbe creare un messaggio da zero, ma di nuovo non può calcolare un rispettivo tag valido senza conoscere $K$.

#### Replay attack e Nonces
Un attacco che il MAC non blocca è invece il **replay attack**: se un attaccante intercetta un messaggio valido $(M, T)$, può semplicemente ritrasmetterlo al destinatario, che lo considererà valido (es. "paga 1000 euro").  

Per contrastare il replay attack bisogna aggiungere al messaggio un valore nuovo per ogni invio, detto **nonce**, incluso nel calcolo del tag:
$$T = \text{MAC}_K(N || M) \quad \text{invia } (N, M, T)$$
Il nonce quindi può essere in chiaro, in quanto la sua funzione è solo permettere al destinatario di controllare che l'invio sia **nuovo** oltre che autentico. 

Tuttavia la semplice aggiunta del nonce alla formula non risolve automaticamente il replay: il destinatario deve avere una regola affidabile per decidere se quel nonce è già stato usato:
- **Nonce incrementale**: il vantaggio è che il destinatario può semplicemente memorizzare l'ultimo nonce ricevuto e accettare solo quelli maggiori. Il problema è che dopo un riavvio, se il contatore riparte da zero con la stessa chiave, si rischiano riutilizzi avversariali;
- **Nonce casuale**: il vantaggio è che non richiede un contatore persistente, lo svantaggio è che per riconoscere se è già stato usato si dovrebbe memorizzare la storia dei nonce ricevuti;
- **Nonce timestamp**: il vantaggio è che permette di verificare se il messaggio appartiene a un intervallo temporale ammissibile, lo svantaggio è che non c'è sempre garanzia che l'orologio del dispositivo e la fonte dell'ora siano affidabili

Un modo intuitivo per capire il problema del timestamp è facendo riferimento al paper del prof sul TOTP (tipo Microsoft Authenticator): se l'applicazione usa l'orologio locale per generare il codice, l'attaccante che riesce ad avere accesso al dispositivo può cambiare l'orologio nel futuro per raccogliere i codici destinati ad essere validi più avanti nel tempo.

Importante saper distinguere tra **MAC** e **firma digitale**. Entrambi servono per proteggere l'integrità e l'autenticità dei messaggi, tuttavia con MAC mittente e destinatario condividono la stessa chiave ed entrambi possono generare un tag valido. Se un terzo osserva una coppia valida allora sa che proviene da qualcuno che conosce la chiave ma non può dimostrare quale dei due l'abbia creata.  
Al contrario con una firma digitale **solo chi firma possiede la chiave privata**: gli altri verificano usando la sua chiave pubblica. Ciò consente una prova più forte dell'origine del messaggio.

Per queste ragioni inoltre se un gruppo possiede una chiave MAC, allora un membro può produrre messaggi fingendosi un altro: un MAC condiviso prova l'appartenenza del gruppo a coloro che possiedono la chiave ma non identifica chi ha scritto il messaggio. 

A volte si farà informalmente riferimento al tag come "firma" di un messaggio, ma è importante ricordare che non si tratta di una firma digitale.

#### Sicurezza del MAC (Unforgeability)
Si presenta la proprietà formale della **unforgeability**, i.e. la resistenza alla falsificazione. 

Si immagina che l'attaccante abbia a disposizione molte coppie valide:
$$(M_1, T_1), (M_2, T_2), \dots$$
Nel modello più forte può anche scegliere adattivamente i messaggi per cui richiedere i tag, per cui decide il messaggio successivo dopo aver visto le risposte ai precedenti. Nonostante tutto questo vantaggio, l'attaccante **non deve essere in grado di produrre un nuovo messaggio valido** $(M^*, T^*)$ per un nuovo messaggio $M^*$ che non ha mai richiesto prima, salvo con probabilità trascurabile. 

Ipotizziamo una funzione ideale: l'attaccante non sa indovinare il tag meglio che tirando a indovinare. Se il tag però ha lunghezza solo **1 byte**, allora un tentativo casuale riesce con probabilità 1/256, che non è assolutamente trascurabile.

**Esercizio Esame**: 

<img src="imgs/consegna.png" width="400" />
<img src="imgs/svolg1.png" width="400" />
<img src="imgs/svolg2.png" width="400" />

### **Rottura di Integrity in WEP**
Il CRC-32 è un checksum di 32 bit utile a rilevare errori accidentali di trasmissione, ed era utilizzato in WEP per proteggere l'integrità dei messaggi (NB non è un MAC, in quanto non richiede una chiave segreta). 

Semplificando, WEP prende il messaggio $M$, calcola il suo CRC $c(M)$, affianca i due valori e li cifra con lo stream cipher:
$$C = [M || c(M)] \oplus S$$
dove si ricorda $S = \text{keystream} = RC4(K || IV)$.

L'idea per cui "anche se il CRC non è crittografico, si trova dentro la cifratura e quindi un attaccante non potrà modificarlo correttamente" è falsa come si vede dal seguente attacco di modifica.  
L'attaccante sceglie una sequenza $\Delta$ di bit della stessa lunghezza del messaggio cifrato: i bit a 1 in $\Delta$ indicano quali bit del ciphertext vuole invertire (se era 0 nel ciphertext con 1 diventa 1, se era 1 con 1 diventa 0). Senza conoscere né $M$ né il keystream $S$, l'attaccante calcola il CRC di $\Delta$, cioè $c(\Delta)$, e ottiene quindi la sequenza $[\Delta || c(\Delta)]$ per poi farne lo XOR con il ciphertext del messaggio originale. 

Il ricevitore, dopo la decifratura, ottiene quindi il messaggio modificato $M \oplus \Delta$ e il CRC modificato $c(M) \oplus c(\Delta)$. Per la linearità rispetto allo XOR, si ha che $c(M) \oplus c(\Delta) = c(M \oplus \Delta)$, per cui il CRC che accompagna il messaggio modificato è valido!

Del resto il CRC-32 rileva con alta probabilità errori casuali, ma in questo caso l'attaccante non sta provocando un errore casuale. Si rompe così l'integrità del messaggio, e il destinatario non si accorge della modifica.

Inoltre se l'attaccante ha recuperato una coppia IV-keystream come nella lezione precedente, può scegliere un nuovo messaggio $M'$, calcolarne pubblicamente il CRC e produrre
$$C^* = [M^* || c(M^*)] \oplus S$$
Trasmettendo questo ciphertext con l'IV associato a $S$, costruisce un frame che supera il controllo. Qui i problemi quindi si combinano: non solo il riutilizzo del keystream aiuta a produrre ciphertext da zero, ma il CRC senza chiave non fornisce alcuna autenticazione.

Quindi in generale:
1. un controllo per la sicurezza deve essere progettato contro modifiche volontarie, non limitarsi a rilevare errori casuali;
2. l'integrità crittografica richiede una costruzione autenticata che dipenda da una chiave segreta. Mettere un checksum ordinario in una cifratura non lo trasforma in un MAC.